# Crab SDK 远程接口全功能教程 (Notebook 版)

**Crab** 是一个专为 LLM Agent / 长时任务设计的沙箱运行时，提供 Docker 级别的隔离 +
CRIU 级别的 checkpoint/restore + fork/txn 等高级原语，并通过 HTTP Gateway 暴露一套
完整的 SDK 接口，支持从远程客户端管理沙箱、执行命令、观测文件系统变化等。

本 notebook 演示 Crab SDK 暴露的全部远程操作，包括公共镜像按需拉取、运行时 baseline、
daemon 强制执行超时、网络 tri-state、富返回值 `ActionResult`、异步 checkpoint / changeset、
inspector 只读 peek，以及 Sandbox 级 `auto_checkpoint` 模式。每个功能一个 cell，可独立运行，
在 cell 之间保持，方便交互式探索。

## 前置条件

1. **部署 Gateway**：确保目标机器已部署 `crab-gateway`（或 `crabd`）并监听 HTTP 端口。
   参考 [`docs/dev-quickstart.md`](../../docs/dev-quickstart.md) 或
   [`tools/vm/provision-service-vm.sh`](../../tools/vm/provision-service-vm.sh)。
2. **安装 SDK**：
   ```bash
   pip install crab           # 从 PyPI
   # 或在项目根目录：
   pip install -e .
   ```
3. **准备凭证**：在 Gateway 上创建一个 tenant 并生成 API key。
4. **修改下方的 `GATEWAY_URL` 和 `API_KEY`**（下一个 cell）。

## 运行方式

```bash
jupyter notebook examples/sdk/04_remote_service_tutorial.ipynb
# 或 VS Code / JupyterLab 打开
```

每个 cell 建议按顺序执行；若中间某步失败可以修 bug 后单独重跑那个 cell，
不必从头开始（这就是 notebook 相较 `.py` 脚本的最大优势）。本 notebook 只回收自己
创建的 sandbox，不会删除同一 tenant 中已有的 workload。


## ⚙️ 用户配置区

通过**环境变量**提供 Gateway 地址和 API key，不要把真实凭证写死进 notebook：

```bash
export CRAB_GATEWAY_URL=http://your-gateway-host:8900
export CRAB_API_KEY=crab_sk_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
export CRAB_TUTORIAL_IMAGE=python:3.12-slim  # 可选；未缓存时自动 pull
export CRAB_TUTORIAL_NETWORK=auto            # auto | host | isolated
```

然后在启动 Jupyter 前 `export`，或在下方 cell 里用 `os.environ` 读取。其他 cell 依赖这两个值。


In [ ]:
# ============================================================
# ★ 用户配置区 ★  —— 通过环境变量提供凭证（勿硬编码真实 key/URL）
# ============================================================
import os
import uuid

# export CRAB_GATEWAY_URL=http://your-gateway-host:8900
# export CRAB_API_KEY=crab_sk_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
GATEWAY_URL = os.environ.get("CRAB_GATEWAY_URL", "http://YOUR_GATEWAY_HOST:8900")
API_KEY = os.environ.get("CRAB_API_KEY", "YOUR_API_KEY")

if API_KEY == "YOUR_API_KEY" or GATEWAY_URL == "http://YOUR_GATEWAY_HOST:8900":
    print("请先设置环境变量: export CRAB_GATEWAY_URL=... CRAB_API_KEY=...")

def network_setting():
    raw = os.environ.get("CRAB_TUTORIAL_NETWORK", "auto").strip().lower()
    if raw in {"auto", "default", "null", "none"}:
        return None
    if raw in {"host", "false", "off", "0"}:
        return False
    if raw in {"isolated", "true", "on", "1"}:
        return True
    raise ValueError("CRAB_TUTORIAL_NETWORK must be auto, host, or isolated")

# 非敏感配置。默认资源保持较小，避免挤占同 tenant 的真实 workload。
SANDBOX_IMAGE = os.environ.get("CRAB_TUTORIAL_IMAGE", "ubuntu:22.04")
SANDBOX_NETWORK = network_setting()
SANDBOX_RESOURCES = {
    "memory": os.environ.get("CRAB_TUTORIAL_MEMORY", "512M"),
    "cpus": int(os.environ.get("CRAB_TUTORIAL_CPUS", "1")),
}
AUX_SANDBOX_MEMORY = os.environ.get("CRAB_TUTORIAL_AUX_MEMORY", "256M")
RUN_APT_CHECK = os.environ.get("CRAB_TUTORIAL_RUN_APT", "1").strip().lower() not in {"0", "false", "no", "off"}
TUTORIAL_RUN_ID = os.environ.get("CRAB_TUTORIAL_RUN_ID", uuid.uuid4().hex[:8])

def tutorial_name(role):
    return f"crab-tutorial-{TUTORIAL_RUN_ID}-{role}"

# 全局：记录本 notebook 创建的所有沙箱，最后一个 cell 统一清理
sandboxes_to_kill = []
created_ids = set()

print(f"配置就绪，Gateway: {GATEWAY_URL}")


## 1️⃣ 连接 Gateway

`Engine.connect(url=..., api_key=...)` 会返回一个 `RemoteEngine` 实例——它是
本地 `Engine` 的完全 API 兼容代理，把所有调用透明地转发到 Gateway。

- 底层是普通的 `requests.Session`，鉴权走 `Authorization: Bearer <api_key>` header。
- Gateway 会根据 API key 做 tenant 隔离，你只能看到自己 tenant 下的沙箱。
- 后续所有 cell 都复用这个 `engine` 变量。


In [ ]:
import time
t0 = time.time()
from crab import Engine, Sandbox
from crab.errors import SandboxExecTimeout
from crab.models import ExecEvent, ExecDone

engine = Engine.connect(url=GATEWAY_URL, api_key=API_KEY)
print(f"Engine 连接成功，类型: {type(engine).__name__}")
print(f"⏱ {time.time()-t0:.3f}s")

## 🛡️ 记录已有沙箱（保留现有 workload）

这里只记录运行前已经存在的 sandbox，绝不删除它们。最后的清理 cell 只遍历本 notebook
成功创建并登记到 `sandboxes_to_kill` 的对象，并再次确认已有 workload 仍然存在。

In [ ]:
import time
t0 = time.time()
existing = engine.list_sandboxes()
preexisting_ids = {
    str(row["sandbox_id"]) for row in existing if row.get("sandbox_id")
}
print(f"发现并保留 {len(preexisting_ids)} 个已有沙箱")
print(f"⏱ {time.time()-t0:.3f}s")

## 2️⃣ 远程创建沙箱

`Sandbox(image=..., resources=..., engine=engine)` 会：

1. 客户端预分配一个 `sbx-<uuid[:12]>` 形式的 sandbox id；
2. 通过 Gateway 转发到 daemon 的 `POST /sandboxes/launch`；
3. daemon 在 cache miss 时先 pull 公共 Docker Hub image，再完成 export 和 OCI bundle；
4. clone 后写入 DNS/capability baseline，并按 `network=True/False/None` 解析网络模式；
5. 返回 running 状态的沙箱句柄及不可变 image digest、baseline schema、有效网络模式。

首次拉取 image 可能耗时 10~30s；之后同一 image 会命中本地缓存。


In [ ]:
import time
t0 = time.time()
sandbox = Sandbox(
    image=SANDBOX_IMAGE,
    resources=SANDBOX_RESOURCES,
    network=SANDBOX_NETWORK,
    name=tutorial_name("main"),
    engine=engine,
)
sandboxes_to_kill.append(sandbox)
created_ids.add(str(sandbox.sandbox_id))
print(f"沙箱创建成功, id = {sandbox.sandbox_id}")
metadata = sandbox.describe().metadata
for key in ("image_reference", "image_id", "image_digest", "rootfs_preparation_schema", "network_mode"):
    print(f"  {key}: {metadata.get(key)}")
assert all(metadata.get(key) for key in ("image_reference", "image_id", "image_digest", "rootfs_preparation_schema", "network_mode"))
print(f"⏱ {time.time()-t0:.3f}s")

## 2️⃣.a Runtime baseline：DNS、capability 与普通 APT

新建 bare-image sandbox 应立即拥有可用 DNS，并使用明确的非特权 capability profile：
`CAP_SETGID` / `CAP_SETUID` 可用，`CAP_SYS_ADMIN` 不可用。Ubuntu/Debian 系镜像还应能让
APT 正常切换到 `_apt` 用户，无需设置 `APT::Sandbox::User=root`。


In [ ]:
baseline = sandbox.commands.run(
    """set -eu
test -s /etc/resolv.conf
dns_answer=$(getent hosts archive.ubuntu.com | head -n 1)
cap_hex=$(awk '/^CapEff:/{print $2}' /proc/self/status)
cap_dec=$(printf '%d' \"0x${cap_hex}\")
test $((cap_dec & (1 << 6))) -ne 0
test $((cap_dec & (1 << 7))) -ne 0
test $((cap_dec & (1 << 21))) -eq 0
printf 'dns=%s\nCapEff=%s\n' \"$dns_answer\" \"$cap_hex\"
""",
    timeout=60.0,
    check=True,
)
print(baseline.stdout)
if RUN_APT_CHECK:
    apt = sandbox.commands.run(
        "DEBIAN_FRONTEND=noninteractive apt-get update -qq && "
        "DEBIAN_FRONTEND=noninteractive apt-get install -y -qq curl && "
        "curl --version | head -n 1",
        timeout=240.0,
        check=True,
    )
    print(apt.stdout)


## 2️⃣.b Exec hard timeout

`timeout` 是 daemon 强制执行的硬边界。收到 `SandboxExecTimeout` 时，命令及其 descendants
已经被完整回收；不会只杀掉 host 侧的 `runc exec` client，也不会影响 sandbox init 或其他 exec。


In [ ]:
try:
    sandbox.commands.run(
        "echo $$ > /tmp/crab-tutorial-timeout-shell.pid; "
        "sleep 30 & echo $! > /tmp/crab-tutorial-timeout-child.pid; wait",
        timeout=1.0,
    )
    raise AssertionError("超时命令意外正常返回")
except SandboxExecTimeout as exc:
    print(f"收到稳定的 SandboxExecTimeout: timeout={exc.timeout}s")

reaped = sandbox.commands.run(
    """set -eu
for pid_file in /tmp/crab-tutorial-timeout-shell.pid /tmp/crab-tutorial-timeout-child.pid; do
    pid=$(cat \"$pid_file\")
    ! kill -0 \"$pid\" 2>/dev/null
done
echo reaped
""",
    timeout=10.0,
    check=True,
)
print(reaped.stdout)


## 3️⃣ 列出当前 tenant 的沙箱

`engine.list_sandboxes()` 走 `GET /sandboxes` 路由，返回当前 API key 可见的所有沙箱条目
（Gateway 会自动按 tenant 过滤，看不到别的租户的沙箱）。

返回值是 `list[dict]`，每个 dict 至少包含 `sandbox_id`、`status`；具体字段随 daemon 版本
而定，dict 遵循开放式 schema。


In [ ]:
import time
t0 = time.time()
sandboxes = engine.list_sandboxes()
print(f"当前沙箱列表: {len(sandboxes)} 个")
for s in sandboxes:
    print(f"  {s.get('sandbox_id')} - {s.get('status', '?')}")
print(f"⏱ {time.time()-t0:.3f}s")

## 4️⃣ 执行命令 (`commands.run`)

最基础的用法：`sandbox.commands.run(cmd)` 同步在沙箱内执行 shell 命令，返回
`ActionResult`（其实是 `subprocess.CompletedProcess` 的超集）。

- `.returncode`：进程退出码；
- `.stdout` / `.stderr`：stdout/stderr 全量文本；
- 未启用富返回值特性时，`.checkpoint` / `.changeset` / `.filesystem_changed` / `.process_changed` 都是 `None`。


In [ ]:
import time
t0 = time.time()
result = sandbox.commands.run("echo hello && uname -a && whoami")
print(f"返回码: {result.returncode}")
print(f"stdout:\n{result.stdout.rstrip()}")
print(f"⏱ {time.time()-t0:.3f}s")

## 5️⃣ 富返回值 — `checkpoint=True` + `observe=True`

`commands.run(cmd, checkpoint=True, observe=True)` 触发两个附加行为：

### AsyncCheckpoint（异步 checkpoint）
- 客户端**预先**分配一个 `ckpt-<uuid[:12]>` id，`run()` **立刻返回**，`result.checkpoint`
  是 `AsyncCheckpoint` 句柄，`.checkpoint_id` 可以立即读到。
- 这个 id 是稳定的 **logical checkpoint id**。后台由 inspector 决定复用旧物理状态、只做 filesystem checkpoint，还是做 process + filesystem checkpoint；调用 `.wait(timeout=...)` 阻塞取回同一个 id，或调用
  `.done` 属性做非阻塞判断。
- 这样设计的好处：agent 可以立刻读取 stdout 并在客户端做下一步决策；同一 sandbox 的下一条命令
  会通过 backpressure 等待 checkpoint 完成，避免污染正在捕获的状态。需要 restore/fork 时，`.wait()` 能保证已经落盘。

### Observe（只读 inspector peek）
- `observe=True` 触发 daemon 端的只读 inspector 查询（`GET /sandboxes/{id}/inspector`）。
- `.filesystem_changed` / `.process_changed` 返回 `True`/`False`（**不再是 `None`**，
  见 `fix(sdk): make ActionResult changeset/observe work on direct sandboxes`）。
- Peek 不重置任何游标，可以放心多次调用。
- **语义：** `filesystem_changed` / `process_changed` 表示 “本次 action 是否改变了状态”。SDK 在 *启动后台 checkpoint 之前* 就完成 peek，因此读到的是 checkpoint 重置游标之前的干净状态。checkpoint 完成后会触发 `host-inspector.reset()` 把两个游标清零；若 peek 排在 checkpoint 之后，会与该 reset 竞争而错误地报 False（历史 bug，已修复；对应回归测试 `test_peek_runs_before_async_checkpoint_reset`）。所以下面这个 mutating 命令会稳定看到 `filesystem_changed=True`。


In [ ]:
import time
t0 = time.time()
result = sandbox.commands.run(
    "echo 'hello' > /tmp/observed.txt && mkdir -p /opt/new",
    checkpoint=True,
    observe=True,
)
exec_latency = time.time() - t0
print(f"返回码: {result.returncode}  (exec+observe 返回, 不含 checkpoint 开销)")
print(f"exec 返回延迟: {exec_latency:.3f}s")
print(f"checkpoint_id (预分配): {result.checkpoint.checkpoint_id}")
print(f"checkpoint 完成?: {result.checkpoint.done}")   # 期望 False（daemon 后台异步执行中）
print(f"filesystem_changed: {result.filesystem_changed}")
print(f"process_changed:    {result.process_changed}")

# 阻塞等待后台 checkpoint 完成（轮询 daemon 的 jobs 端点）
t1 = time.time()
ckpt_id = result.checkpoint.wait(timeout=60.0)
wait_latency = time.time() - t1
print(f"checkpoint 已完成: {ckpt_id}  (wait 轮询延迟: {wait_latency:.3f}s)")
print(f"checkpoint 完成?: {result.checkpoint.done}")   # 期望 True
print(f"sandbox.last_checkpoint_id = {sandbox.last_checkpoint_id}")
checkpoint_info = next((item for item in sandbox.checkpoints.list() if item.get("checkpoint_id") == ckpt_id), None)
if checkpoint_info:
    print(f"materialization: {checkpoint_info.get('materialization')}")
    print(f"physical sources: process={checkpoint_info.get('process_checkpoint_id')}, filesystem={checkpoint_info.get('filesystem_checkpoint_id')}")
print(f"→ exec 返回 {exec_latency:.3f}s + checkpoint 等待 {wait_latency:.3f}s")
print(f"⏱ {time.time()-t0:.3f}s")

## 5️⃣.b 背压 — 上一个后台 checkpoint 阻塞下一个 `run`

daemon 侧 **per-sandbox 背压**：新的 `/action` 到来时，如果该 sandbox 还有后台 checkpoint 在跑，
daemon 会**先等它完成再开始 exec**。用户感知为「上一个 checkpoint 慢了，这次请求就稍等一等」——
延迟（而非丢弃）请求。等待有 600s 上限，超时则记 warning 并照常 exec，避免卡死请求。

**如何隔离出背压带来的额外延迟**（否则会被 exec 本身耗时淹没）：
- `run#A`：触发一个后台 checkpoint（不等待）——制造一个 pending checkpoint
- `run#B`：紧跟 `run#A` 发起 → 其 exec 返回时间**包含**等待 `run#A` checkpoint 的背压时间
- `run#C`：等到完全空闲后再发起 → 无背压，作为**基线**
- 背压额外延迟 ≈ `run#B` 延迟 − `run#C` 延迟

> **重要事实：** logical checkpoint 会由 inspector 自适应决定物理工作。本段 run#A 写了文件，
> 所以至少生成 filesystem checkpoint；ZFS 快照是 O(1) 写时复制。无变化轮次则可完全复用旧物理状态。
> 背压逻辑本身由单元测试 `tests/test_daemon_action_backpressure.py` 注入 1s 延迟做了严格验证。


In [ ]:
import time
# run#A：制造一个 pending 后台 checkpoint（创建一批文件后 checkpoint，不等待）
sandbox.commands.run(
    "mkdir -p /opt/many && cd /opt/many && "
    "for i in $(seq 1 10240); do echo x > f$i; done",
    checkpoint=True,
)
print("run#A 已触发后台 checkpoint（不等待，制造 pending 状态）")

# run#B：紧跟发起 -> 被 run#A 的后台 checkpoint 背压阻塞
t1 = time.time()
result_blocked = sandbox.commands.run("echo blocked", checkpoint=True)
blocked_latency = time.time() - t1
print(f"run#B (紧跟 run#A) exec 返回: {blocked_latency:.3f}s (含背压等待)")
result_blocked.checkpoint.wait(timeout=120.0)  # 等到完全空闲

# run#C：空闲基线 -> 无 pending checkpoint，无背压
t2 = time.time()
result_idle = sandbox.commands.run("echo idle", checkpoint=True)
idle_latency = time.time() - t2
print(f"run#C (空闲基线) exec 返回: {idle_latency:.3f}s (无背压)")
result_idle.checkpoint.wait(timeout=120.0)

delta_ms = (blocked_latency - idle_latency) * 1000.0
print(f"→ 背压额外延迟 ≈ {delta_ms:.0f}ms "
      f"(run#B {blocked_latency:.3f}s - run#C {idle_latency:.3f}s)")
if blocked_latency > idle_latency:
    print("✓ 背压生效：紧跟的 run#B 比空闲的 run#C 慢（等待了上一个 checkpoint）")
else:
    print("(注: ZFS 快照 O(1)，checkpoint ~0.1s，背压额外延迟很小/接近噪声)")


## 6️⃣ 富返回值 — `changeset` 异步 / 同步

`commands.run(cmd, changeset=True [, changeset_sync=True])` 追加返回本次命令引入的文件变更：

- `changeset=True`（默认异步）：`result.changeset` 是 `AsyncChangeset`，后台线程计算，
  `.wait(timeout=...)` 拿 `list[dict]`。
- `changeset_sync=True`：同步计算，`result.changeset` 直接是 `list[dict]`。

### "since 上一个 checkpoint" 语义

SDK 会自动用**上一个完成的 checkpoint**（`sandbox.last_checkpoint_id`，此处是 step 5 建的那个）
作为 `since` 参数：changeset 只 diff *从上次 checkpoint 到本次 run 结束* 之间的变更，
不包含更早的历史。这一设计还避免了对普通（非 fork）沙箱走 `fork_changeset` 路径而 400 报错
（`fork_changeset` 仅对 fork 出来的沙箱可用）。

如果 step 5 还没跑（`last_checkpoint_id` 为 `None`），下面这个 cell 会回退到
`fork_changeset` 路径——请先跑 step 5 再跑这里。


In [ ]:
import time
t0 = time.time()
# 6a. 异步 changeset
result = sandbox.commands.run("touch /tmp/async_test", changeset=True)
print(f"result.changeset 类型: {type(result.changeset).__name__}")
entries = result.changeset.wait(timeout=60.0)
print(f"异步 changeset: {len(entries)} 条变更")
for entry in entries[:5]:
    print(f"  {entry}")
print(f"⏱ {time.time()-t0:.3f}s")

In [ ]:
import time
t0 = time.time()
# 6b. 同步 changeset
result = sandbox.commands.run(
    "rm /tmp/async_test", changeset=True, changeset_sync=True
)
print(f"result.changeset 类型: {type(result.changeset).__name__}")
assert isinstance(result.changeset, list), "同步模式应直接返回 list"
print(f"同步 changeset: {len(result.changeset)} 条变更")
for entry in result.changeset[:5]:
    print(f"  {entry}")
print(f"⏱ {time.time()-t0:.3f}s")

## 7️⃣ `auto_checkpoint` 模式

`Sandbox(..., auto_checkpoint=True)` 让沙箱进入"每次 `commands.run` 后自动做后台 checkpoint"
的模式：

- 每次 `run` 返回时都已经悄悄触发了一个 `AsyncCheckpoint`；
- `sandbox.last_checkpoint_id` 始终是最新的 logical checkpoint id，可以直接读；
- inspector 自动选择不落新物理数据、filesystem-only、或 process + filesystem；
- 无需每次都手写 `checkpoint=True`。

典型场景：LLM Agent 长任务，每一步都想留 checkpoint 以便万一崩了可以 restore，
但又不想每步都写模板代码。


In [ ]:
import time
t0 = time.time()
auto_sb = Sandbox(
    image=SANDBOX_IMAGE,
    resources={"memory": AUX_SANDBOX_MEMORY},
    network=SANDBOX_NETWORK,
    name=tutorial_name("auto"),
    engine=engine,
    auto_checkpoint=True,
)
sandboxes_to_kill.append(auto_sb)
created_ids.add(str(auto_sb.sandbox_id))
print(f"auto_checkpoint 沙箱创建成功: {auto_sb.sandbox_id}")

auto_sb.commands.run("echo step1 > /tmp/s1")
print(f"第一次自动 checkpoint: {auto_sb.last_checkpoint_id}")

# 后台进程演示：detach=True 让 SDK 把命令 stdio 重定向到 /dev/null
# （等价于前置 `exec 1>/dev/null 2>&1;`），解除 `runc exec` 的管道继承阻塞——
# 否则光加 `&` 会一直阻塞到进程退出。detach 只解除“管道继承阻塞”，
# 不改变进程生命周期：命令自身仍须用 `&` 后台化 run 才会立即返回；
# detach 模式不返回输出。这个 sleep 30 后台进程会被本次 auto_checkpoint 捕获。
# observe=True 在 exec 之后、checkpoint 之前做一次只读 inspector peek。
# process_changed 由 host-inspector 对比“当前 cgroup 活进程集”与“上次
# checkpoint 基线”得出（实时读 cgroup PID，不是靠消费 eBPF 事件），所以
# 只要后台进程还活着 process_changed 就稳定为 True；本命令没写磁盘，
# 因此 filesystem_changed=False（VM 实测：proc=True / fs=False）。
tmp1 = auto_sb.commands.run(
    "sleep 30 & sleep 2", detach=True, observe=True
)
print(f"后台进程已启动，自动 checkpoint: {auto_sb.last_checkpoint_id}")
print(f"  filesystem_changed: {tmp1.filesystem_changed}")
print(f"  process_changed: {tmp1.process_changed}")

tmp2 = auto_sb.commands.run("echo step2 > /tmp/s2", observe=True)
print(f"第二次自动 checkpoint: {auto_sb.last_checkpoint_id}")
print(f"  filesystem_changed: {tmp2.filesystem_changed}")
print(f"  process_changed: {tmp2.process_changed}")
print(f"⏱ {time.time()-t0:.3f}s")

## 8️⃣ 流式执行 (`commands.stream`)

`commands.stream(cmd)` 返回一个生成器，实时 yield 沙箱内进程的 stdout/stderr 行：

- `ExecEvent(channel, text)`：一行输出（`channel` 是 `"stdout"` 或 `"stderr"`）；
- `ExecDone(returncode)`：进程退出，标记流的结束。

Gateway 使用 chunked HTTP streaming 转发，延迟接近 daemon 直连。
适合长时任务、需要实时看进度或做流式解析的场景（如 tail -f、npm build 等）。


In [ ]:
import time
t0 = time.time()
print("命令: for i in 1 2 3; do echo step-$i; sleep 0.5; done")
print("输出:")
for event in sandbox.commands.stream(
    "for i in 1 2 3; do echo step-$i; sleep 0.5; done"
):
    if isinstance(event, ExecEvent):
        print(f"  [{event.channel}] {event.text}", end="")
    elif isinstance(event, ExecDone):
        print(f"  [exit] returncode={event.returncode}")
print(f"⏱ {time.time()-t0:.3f}s")

## 9️⃣ 文件变更 — `sandbox.changeset(since, force=True)`

除了 step 6 里 `commands.run(..., changeset=True)` 的富返回值形式，还可以显式调用
`sandbox.changeset(since_checkpoint_id, force=True)` 主动查询：

- 需要给出 `since=<某个 checkpoint id>` 作为 diff 基线；
- `force=True` 跳过 daemon 的 inspector gate 优化——不启用 eBPF host inspector 时，
  gate 会保守地判定"无变更"而返回空列表；`force=True` 强制走 diff 逻辑，拿到真实变更集。

流程：先建基线 checkpoint → 写文件 → 查 changeset。


In [ ]:
import time
t0 = time.time()
# 先做一次 checkpoint 作为 changeset 的基线
base_ckpt = sandbox.checkpoint("changeset-base")
print(f"基线 checkpoint: {base_ckpt}")

# 写入测试文件
sandbox.commands.run("echo 'hello changeset' > /tmp/changeset_test.txt")
sandbox.commands.run("mkdir -p /opt/demo && echo 123 > /opt/demo/data.txt")
print("写入了 /tmp/changeset_test.txt 和 /opt/demo/data.txt")

# force=True: 无需等 inspector 观察到变更即可拿到结果
changes = sandbox.changeset(base_ckpt, force=True)
print(f"changeset 返回 {len(changes)} 条变更 (force 跳过 inspector gate)")
for entry in changes[:10]:
    print(f"  {entry}")
print(f"⏱ {time.time()-t0:.3f}s")

## 🔟 Checkpoint (`sandbox.checkpoint`)

`sandbox.checkpoint(label)` 是同步版本的 checkpoint API：调用后阻塞直到 CRIU dump 完成，
返回 `ckpt-<uuid[:12]>` 形式的 id。这个 id 可以后续用于：

- `sandbox.restore(ckpt_id)` —— 把沙箱回滚到该 checkpoint；
- `sandbox.fork(n, checkpoint_id=ckpt_id)` —— 从该 checkpoint fork 出 n 个副本（父沙箱不回滚）；
- `sandbox.changeset(ckpt_id, ...)` —— 查询"从此 checkpoint 至今"的文件变更。

**依赖**：daemon 主机需要装 CRIU 3.17+ 并且有 `CAP_SYS_ADMIN`。如果没有，会抛异常。


In [ ]:
import time
t0 = time.time()
ckpt_id = sandbox.checkpoint("tutorial-ckpt")
print(f"Checkpoint 创建成功: {ckpt_id}")
print(f"⏱ {time.time()-t0:.3f}s")

## 1️⃣1️⃣ Fork (`sandbox.fork(n)`)

`sandbox.fork(n)` 从当前沙箱（自动先 checkpoint）派生出 `n` 个**完全隔离**的副本：

- 每个 fork 有自己独立的 sandbox id、独立的进程树、独立的 rootfs（ZFS clone）；
- fork 内的修改**不会**影响原沙箱，反之亦然；
- Fork 是 CRIU restore + ZFS clone 的组合，语义上等价于 UNIX `fork()`——瞬时快照，随后独立演化。

典型用途：speculative execution、并行 A/B 实验、并行 grader 打分等。

下面这个 cell 会：
1. Fork 1 个副本；
2. 在 fork 里创建一个"标记文件"；
3. 回到原沙箱验证它**没有**这个文件（隔离性证明）。


In [ ]:
import time
t0 = time.time()
forks = sandbox.fork(1)
fork_sbx = forks[0]
sandboxes_to_kill.append(fork_sbx)
created_ids.add(str(fork_sbx.sandbox_id))
print(f"Fork 成功, fork id = {fork_sbx.sandbox_id}")
source_metadata = sandbox.describe().metadata
fork_metadata = fork_sbx.describe().metadata
assert fork_metadata.get("network_mode") == source_metadata.get("network_mode")
if source_metadata.get("network_mode") == "isolated":
    assert fork_metadata.get("network_namespace_path") != source_metadata.get("network_namespace_path")

# 在 fork 里做一些修改
fork_result = fork_sbx.commands.run(
    "echo 'I am a fork' > /tmp/fork_marker.txt && cat /tmp/fork_marker.txt"
)
print(f"Fork 内执行结果: {fork_result.stdout.rstrip()}")

# 验证原沙箱没有这个文件（隔离性）
orig_check = sandbox.commands.run("cat /tmp/fork_marker.txt 2>&1 || true")
print(f"原沙箱检查 (应找不到文件): {orig_check.stdout.rstrip()}")
print(f"⏱ {time.time()-t0:.3f}s")

## 1️⃣1️⃣.b 从历史 checkpoint fork (`sandbox.fork(n, checkpoint_id=...)`)

上面的 `sandbox.fork(n)` 从沙箱的**当前活状态**分叉。传入 `checkpoint_id` 则从一个**已存在的 checkpoint**
分叉，语义差别很实在：

- 不会新打 checkpoint —— 直接克隆那个历史快照；
- 父沙箱**不会被 restore**，它继续在原处运行（这不是 `restore` + `fork`）；
- `count > 1` 时，整批 fork 都从同一个历史点分叉。

典型用途：从一个已知良好的状态反复重跑（RL rollout / tree search 的回溯），而不打断正在前进的主时间线。

下面这个 cell 用 step 🔟 的 checkpoint 做时间锚点：
1. 在父沙箱写一个「checkpoint 之后才存在」的标记文件；
2. 从该 checkpoint fork —— fork 里**不应该**有这个文件（回到了过去）；
3. 父沙箱**仍然**有这个文件（证明父沙箱没有被回滚）。

验证完立刻 kill 掉这个 fork：每个沙箱都占 tenant 配额，后面的步骤还要用。


In [ ]:
import time
t0 = time.time()
# 1) 在 step 🔟 的 checkpoint 之后写一个标记文件
sandbox.commands.run("echo 'after ckpt' > /tmp/after_ckpt.txt")

# 2) 从那个历史 checkpoint fork
past_fork = sandbox.fork(1, checkpoint_id=ckpt_id)[0]
created_ids.add(str(past_fork.sandbox_id))
print(f"Fork 成功, fork id = {past_fork.sandbox_id} (fork 点 = {ckpt_id})")

in_fork = past_fork.commands.run("cat /tmp/after_ckpt.txt 2>&1 || true")
print(f"fork 内检查 (应找不到该文件): {in_fork.stdout.rstrip()}")

# 3) 父沙箱没有被回滚，文件仍在
in_parent = sandbox.commands.run("cat /tmp/after_ckpt.txt 2>&1 || true")
print(f"父沙箱检查 (文件应仍在): {in_parent.stdout.rstrip()}")

past_fork.kill()  # 立刻回收，给后续步骤留出配额
print(f"已回收历史 fork: {past_fork.sandbox_id}")
print(f"⏱ {time.time()-t0:.3f}s")


## 1️⃣2️⃣ Transaction — `begin` / `exec` / `commit` | `abort`

Transaction 是"可回滚的沙箱操作序列"：

- `txn = sandbox.begin(label)` 开启一个事务（内部隐式建一个基线 checkpoint）；
- `txn.exec(cmd)` 在事务里执行命令——所有修改都被跟踪；
- `txn.commit()` 提交事务：修改落地到主线，返回 `TxnCommitResult`；
- `txn.abort()` 放弃事务：沙箱回滚到基线 checkpoint，仿佛什么都没发生过。

Transaction 是 fork 语义的"try/except"版：适合 agent 里 "试探性执行——不满意就撤销" 的场景。


In [ ]:
import time
t0 = time.time()
txn = sandbox.begin("tutorial-txn")
print(f"事务已开启: {txn}")

# 在事务内执行操作
txn_result = txn.exec("echo 'inside txn' > /tmp/txn_file.txt")
print(f"事务内 exec 返回码: {txn_result.returncode}")

# 提交事务
commit_result = txn.commit()
print(f"事务提交成功: {commit_result}")

# 验证事务结果
verify = sandbox.commands.run("cat /tmp/txn_file.txt 2>&1 || true")
print(f"事务提交后验证: {verify.stdout.rstrip()}")
print(f"⏱ {time.time()-t0:.3f}s")

## 1️⃣3️⃣ 连接已有沙箱 (`Sandbox.connect`)

`Sandbox.connect(sandbox_id, engine=engine)` 通过 id 重新绑定到一个 daemon 侧已存在的沙箱——
返回的 `Sandbox` 对象和最初 `Sandbox(...)` 创建时拿到的**等价**。

典型场景：
- 多个 notebook / 脚本共享同一个长期存活的沙箱；
- 上次会话崩了但沙箱在 daemon 侧还活着，重新接上；
- 通过 CI/CD 传递 sandbox id 给下游任务。

**注意**：`connect` 不会创建新沙箱，只是"接线"。id 不存在会抛异常。


In [ ]:
import time
t0 = time.time()
existing_id = sandbox.sandbox_id
print(f"重新连接到: {existing_id}")

reconnected = Sandbox.connect(existing_id, engine=engine)
print(f"重新连接成功, id = {reconnected.sandbox_id}")

# 验证可以执行命令
re_result = reconnected.commands.run("echo 'reconnected!'")
print(f"通过重连沙箱执行命令: {re_result.stdout.rstrip()}")
print(f"⏱ {time.time()-t0:.3f}s")

## 1️⃣4️⃣ 端口暴露 (`ports.expose`)

`sandbox.ports.expose(guest_port)` 在 gateway 主机上分配一个 host_port，
把外部 `host_ip:host_port` 上的 TCP 流量转发到沙箱内部 `guest_port`：

- 底层是 netns 里的 iptables + L4 转发；
- 返回 `PortAllocation`，字段包括 `guest_port` / `host_port` / `url`（`tcp://host_ip:host_port`）；
- 如需 HTTP 层访问，用 `http://<gateway_host>:<host_port>/` 拼接即可。

### VM NAT 限制

如果 gateway 跑在 QEMU/KVM VM 里（例如项目自带的 `provision-service-vm.sh`），
VM 只对宿主机的 `hostfwd` 端口开放了 SSH 和 gateway 端口（如 2223、8900），
`ports.expose` 分配的临时 host_port 未必被 VM NAT 转发到宿主机——从 VM 外部访问会失败。
在生产 bare-metal 部署下则没有这个限制。

下面 cell 会启一个 python HTTP server → 暴露端口 → 尝试从当前网络访问（失败也没关系，
证明 `expose` 本身工作）。


In [ ]:
import time
import urllib.request
from urllib.parse import urlparse

def demonstrate_port_exposure():
    t0 = time.time()
    sandbox.commands.run(
        "python3 -m http.server 8080 --directory /tmp &",
        timeout=3.0,
    )
    time.sleep(1)
    allocation = sandbox.ports.expose(8080)
    print(f"端口暴露成功: {allocation.url}")
    gw_host = urlparse(GATEWAY_URL).hostname or "127.0.0.1"
    http_url = f"http://{gw_host}:{allocation.host_port}/"
    try:
        with urllib.request.urlopen(http_url, timeout=5) as resp:
            body = resp.read(200).decode("utf-8", errors="replace")
            print(f"外部访问成功 (HTTP {resp.status}): {body[:200]}")
    except Exception as fetch_exc:
        print(f"外部访问失败（VM NAT 下可能是预期行为）: {fetch_exc}")
    print(f"⏱ {time.time()-t0:.3f}s")

if sandbox.describe().metadata.get("network_mode") == "isolated":
    demonstrate_port_exposure()
else:
    print("当前为 host 网络模式；ports.expose 仅适用于 isolated netns，跳过。")


## 1️⃣5️⃣ 清理 (`sandbox.kill`)

`sandbox.kill()` 停止并回收沙箱：

- 停 runc container、删 ZFS dataset、清 checkpoint 存储、释放 netns / port；
- daemon 会把 sandbox 从 registry 里删掉，后续对该 id 的操作会 404。

下面这个 cell 会遍历本 notebook 创建的所有沙箱，逐个 `kill()`，然后确认运行前已有的
sandbox 一个都没有被教程删除。这是唯一一个用
`try/except` 兜底的 cell——因为清理路径本身可能失败（网络断了、沙箱已经被别的
路径回收了等），但我们希望**继续尝试**清理剩下的，而不是第一个失败就整个 cell abort。


In [ ]:
for sbx in sandboxes_to_kill:
    try:
        sid = sbx.sandbox_id
        sbx.kill()
        print(f"✓ 已清理沙箱: {sid}")
    except Exception as kill_exc:
        print(f"✗ 清理沙箱失败: {kill_exc}")

remaining = engine.list_sandboxes()
remaining_ids = {
    str(row["sandbox_id"]) for row in remaining if row.get("sandbox_id")
}
missing_preexisting = sorted(preexisting_ids - remaining_ids)
leaked_tutorial = sorted(created_ids & remaining_ids)
assert not missing_preexisting, f"已有 sandbox 消失: {missing_preexisting}"
assert not leaked_tutorial, f"tutorial sandbox 未清理: {leaked_tutorial}"
print(f"✓ 运行前的 {len(preexisting_ids)} 个 sandbox 全部保留")
sandboxes_to_kill.clear()
print("\n清理完成。")


## 🎉 总结 & 下一步

本 notebook 覆盖了 Crab SDK 的远程 sandbox 生命周期与 runtime baseline：

| # | 功能 | 关键 API |
|---|------|---------|
| 1 | 连接 Gateway | `Engine.connect` |
| 2 | 创建沙箱 | `Sandbox(image=..., engine=...)` |
| 2a | 镜像/DNS/capability baseline | `sandbox.describe()` / `commands.run()` |
| 2b | daemon 硬超时 | `commands.run(..., timeout=...)` / `SandboxExecTimeout` |
| 3 | 列出沙箱 | `engine.list_sandboxes()` |
| 4 | 执行命令 | `sandbox.commands.run(cmd)` |
| 5 | 富返回值 checkpoint + observe | `commands.run(..., checkpoint=True, observe=True)` |
| 6 | 富返回值 changeset | `commands.run(..., changeset=True [, changeset_sync=True])` |
| 7 | auto_checkpoint 模式 | `Sandbox(..., auto_checkpoint=True)` |
| 8 | 流式执行 | `sandbox.commands.stream(cmd)` |
| 9 | 主动查询 changeset | `sandbox.changeset(since, force=True)` |
| 10 | 同步 checkpoint | `sandbox.checkpoint(label)` |
| 11 | Fork | `sandbox.fork(n)` |
| 12 | Transaction | `sandbox.begin` / `.exec` / `.commit` / `.abort` |
| 13 | 重连沙箱 | `Sandbox.connect(id, engine=...)` |
| 14 | 端口暴露 | `sandbox.ports.expose(port)` |
| 15 | 清理 | `sandbox.kill()` |

### 进一步阅读

- [`docs/sdk.md`](../../docs/sdk.md) — SDK 完整 API 参考
- [`docs/dev-quickstart.md`](../../docs/dev-quickstart.md) — 本地开发环境搭建
- [`docs/architecture.md`](../../docs/architecture.md) — Gateway / daemon / runtime 分层
- [`docs/daemon.md`](../../docs/daemon.md) — daemon 详解
- [`docs/telemetry.md`](../../docs/telemetry.md) — 可观测性

### 其他 examples

- [`01_basic_sandbox.py`](01_basic_sandbox.py) — 单机本地 daemon 快速上手
- [`02_iflow_replay.py`](02_iflow_replay.py) — trace 回放
- [`03_iflow_minimax_qsort.py`](03_iflow_minimax_qsort.py) — speculative execution 案例

Happy hacking with Crab! 🦀
